<a href="https://colab.research.google.com/github/Saru2248/Fuel/blob/main/Day_10_Fuel_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
#sigmoid function
import numpy as np
z= np.array([-4,-2,0,2,4])
prob = 1 / (1+ np.exp(-z))
print(prob.round(2))

[0.02 0.12 0.5  0.88 0.98]


In [29]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 100

data = pd.DataFrame({
    "hours": rng.integers(1,11,n),
    "sleep": rng.integers(4,10,n),
    "attendance": rng.integers(60,101,n)})

noise = rng.normal(0,4,n)
marks = (10 + 4 * data["hours"] + 2* data["sleep"] + 0.3 * data["attendance"] + noise).round()
data ["passed"] = (marks>= 65).astype(int)

print(data.head())
print(data["passed"].value_counts())

   hours  sleep  attendance  passed
0      1      8          75       0
1      8      5          97       1
2      7      8          80       1
3      5      4          88       0
4      5      8          78       0
passed
1    64
0    36
Name: count, dtype: int64


In [30]:
#Statified split

from sklearn.model_selection import( train_test_split)

feats = ["hours", "sleep","attendance"]
train, test = train_test_split(
    data, test_size = 0.25, random_state = 42, stratify = data ["passed"])
y_train = train ["passed"]
y_test = test["passed"]
print(len(train), len(test))
print(y_test.value_counts().to_dict())

75 25
{1: 16, 0: 9}


In [31]:
# train and predict
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
# Use .fit() instead of calling clf() directly
clf.fit(train[feats], y_train)

pred = clf.predict(test[feats])
print(pred[:10])
print(y_test.values[:10])

[1 1 1 0 1 1 0 1 1 1]
[1 1 1 0 1 1 0 1 1 1]


In [32]:
#probabilites with predict_proba
proba =clf.predict_proba(test[feats])[:,1]
print(proba[:5].round (2))

[0.93 1.   0.79 0.23 0.99]


In [33]:
coefs = pd.Series(clf.coef_[0], index = feats)
print(coefs.round(2))

hours         1.30
sleep         0.85
attendance    0.08
dtype: float64


In [34]:
#predict for a new student
student = pd.DataFrame({
    "hours":[6],"sleep":[8],"attendance":[90]})
print(clf.predict_proba(student).round(2))
print(clf.predict(student))


[[0.01 0.99]]
[1]


In [35]:
#confusion matrix
from sklearn.metrics import( confusion_matrix,accuracy_score)
cm= confusion_matrix (y_test, pred)
acc = accuracy_score(y_test,pred)
print(cm)
print("accuracy:",acc)

[[ 6  3]
 [ 0 16]]
accuracy: 0.88


In [36]:
#Compare with a baseline

from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(
    strategy ="most_frequent")
dummy.fit(train[feats],y_train)
print(dummy.score(test[feats],y_test))

0.64


In [37]:
#accuracy can lie
from sklearn.metrics import recall_score

y_true = np.array([1] * 10+ [0] * 990)
y_pred = np.zeros(1000, dtype=int)
acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true,y_pred)
print("Accuracy:", acc)
print("Recall:", rec)


Accuracy: 0.99
Recall: 0.0


In [38]:
#in code
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# Recreate data and model to ensure variables are defined
rng = np.random.default_rng(42)
n = 100
data = pd.DataFrame({
    "hours": rng.integers(1, 11, n),
    "sleep": rng.integers(4, 10, n),
    "attendance": rng.integers(60, 101, n)
})
noise = rng.normal(0, 4, n)
marks = (10 + 4 * data["hours"] + 2 * data["sleep"] + 0.3 * data["attendance"] + noise).round()
data["passed"] = (marks >= 65).astype(int)

feats = ["hours", "sleep", "attendance"]
train, test = train_test_split(
    data, test_size=0.25, random_state=42, stratify=data["passed"]
)
y_train = train["passed"]
y_test = test["passed"]

clf = LogisticRegression()
clf.fit(train[feats], y_train)
pred = clf.predict(test[feats])

# Calculate and print metrics
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
print("Precision:", round(prec, 2))
print("Recall:", round(rec, 2))
print("F1:", round(f1, 2))

Precision: 0.84
Recall: 1.0
F1: 0.91


In [39]:
# Show step-by-step calculation
from sklearn.metrics import confusion_matrix

# Get confusion matrix components
cm = confusion_matrix(y_test, pred)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix Components:")
print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"True Negatives (TN): {tn}")
print(f"False Negatives (FN): {fn}\n")

# 1. Precision calculation
calculated_precision = tp / (tp + fp)
print(f"Precision Calculation: TP / (TP + FP)")
print(f"= {tp} / ({tp} + {fp})")
print(f"= {tp} / {tp + fp}")
print(f"= {calculated_precision:.4f} (rounds to {calculated_precision:.2f})\n")

# 2. Recall calculation
calculated_recall = tp / (tp + fn)
print(f"Recall Calculation: TP / (TP + FN)")
print(f"= {tp} / ({tp} + {fn})")
print(f"= {tp} / {tp + fn}")
print(f"= {calculated_recall:.4f} (rounds to {calculated_recall:.2f})\n")

# 3. F1-score calculation
calculated_f1 = 2 * (calculated_precision * calculated_recall) / (calculated_precision + calculated_recall)
print(f"F1-Score Calculation: 2 * (Precision * Recall) / (Precision + Recall)")
print(f"= 2 * ({calculated_precision:.4f} * {calculated_recall:.4f}) / ({calculated_precision:.4f} + {calculated_recall:.4f})")
print(f"= 2 * {calculated_precision * calculated_recall:.4f} / {calculated_precision + calculated_recall:.4f}")
print(f"= {calculated_f1:.4f} (rounds to {calculated_f1:.2f})")

Confusion Matrix Components:
True Positives (TP): 16
False Positives (FP): 3
True Negatives (TN): 6
False Negatives (FN): 0

Precision Calculation: TP / (TP + FP)
= 16 / (16 + 3)
= 16 / 19
= 0.8421 (rounds to 0.84)

Recall Calculation: TP / (TP + FN)
= 16 / (16 + 0)
= 16 / 16
= 1.0000 (rounds to 1.00)

F1-Score Calculation: 2 * (Precision * Recall) / (Precision + Recall)
= 2 * (0.8421 * 1.0000) / (0.8421 + 1.0000)
= 2 * 0.8421 / 1.8421
= 0.9143 (rounds to 0.91)


In [40]:
for th in[0.3,0.5,0.7,0.9]:
    p = (proba >= th).astype(int)
    prc = precision_score(y_test,p)
    rc = recall_score(y_test,p)
    print(th,round (prc,2), round (rc,2))


0.3 0.84 1.0
0.5 0.84 1.0
0.7 0.94 0.94
0.9 1.0 0.88


In [41]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, proba)
print("AUC:", round(auc, 3))

AUC: 0.986


In [42]:
from sklearn.model_selection import (
    cross_val_score)
X,y = data[feats],data ["passed"]
acc = cross_val_score(
    LogisticRegression(),
    X,y,
    cv = 5  )
print(acc.round(2))
print("Mean:", round (acc.mean(),3))


[0.85 1.   0.85 0.85 0.9 ]
Mean: 0.89


In [44]:
#a pipeline and f1 score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipe = make_pipeline(StandardScaler(),
                     LogisticRegression())
f1s = cross_val_score(
    pipe,
    X,y,
    cv = 5,
    scoring = "f1")
print(f1s.round(3))
print("Mean:", round(f1s.mean(),3))

[0.88  0.96  0.923 0.88  0.923]
Mean: 0.913


In [46]:
#compare classifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

model ={
    "logistic": pipe,
    "Tree": DecisionTreeClassifier(
        max_depth = 3,
        random_state = 42
    ),
    "KNN": KNeighborsClassifier()
}
for name, m in model.items():
    s= cross_val_score(
        m,
        X,y,
        cv = 5,
        scoring = "f1"
    )
    print(name, round(s.mean(),3))

logistic 0.913
Tree 0.831
KNN 0.888


In [48]:
import joblib
pipe.fit(X,y)
joblib.dump (pipe,"pass_model.pkl")
loaded = joblib.load("pass_model.pkl")
print(loaded.predict(student))


import os
print(os.listdir())


[1]
['.config', 'pass_model.pkl', 'sample_data']


In [49]:
#say whether each is classification or regression  :loan approved
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Create sample loan dataset
np.random.seed(42)
n = 200
data = pd.DataFrame({
    "income": np.random.randint(30000, 120000, n),
    "credit_score": np.random.randint(550, 850, n),
    "loan_amount": np.random.randint(5000, 50000, n)
})

# Define target: 1 = approved, 0 = rejected based on logical rules + some noise
approved_rule = (data["credit_score"] >= 680) & (data["income"] >= 50000)
data["loan_approved"] = approved_rule.astype(int)

# 2. Define features and target
feats = ["income", "credit_score", "loan_amount"]
X = data[feats]
y = data["loan_approved"]

# 3. Stratified Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 4. Train Logistic Regression Model
clf = LogisticRegression()
clf.fit(X_train, y_train)

# 5. Predict and Evaluate
pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.2f}\n")
print("Classification Report:")
print(classification_report(y_test, pred))

Accuracy: 0.86

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.86      0.87        28
           1       0.83      0.86      0.84        22

    accuracy                           0.86        50
   macro avg       0.86      0.86      0.86        50
weighted avg       0.86      0.86      0.86        50



In [50]:

 # use numpy to compute the sigmoid of 0.3 and -3
import numpy as np

z = np.array([0.3, -3])
prob = 1 / (1 + np.exp(-z))
print(prob.round(2))

[0.57 0.05]


In [51]:
#create a dataframe of 100 rows with  study hours ,attendance and passed(0 or 1)
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)
n = 100

# Create a DataFrame with random values for study hours and attendance
data = pd.DataFrame({
    "study_hours": np.random.randint(1, 11, n),   # Study hours between 1 and 10
    "attendance": np.random.randint(50, 101, n)    # Attendance percentage between 50 and 100
})

# Define a rule to determine if the student passed (1) or failed (0)
# Example rule: passing requires at least 5 study hours and 75% attendance
passed_rule = (data["study_hours"] >= 5) & (data["attendance"] >= 75)
data["passed"] = passed_rule.astype(int)

# Display the first few rows and class counts
print(data.head())
print("\nClass distribution:")
print(data["passed"].value_counts())

   study_hours  attendance  passed
0            7          61       0
1            4          83       0
2            8          82       1
3            5          97       1
4            7          72       0

Class distribution:
passed
0    58
1    42
Name: count, dtype: int64
